In [1]:
from pathlib import Path
import json, pandas as pd, numpy as np

DATA = Path('./data/')

schema = json.loads((DATA / 'feature_schema_context.json').read_text(encoding='utf-8'))
df = pd.read_parquet(DATA / 'model_features_context.parquet')
print(df.shape)                            # (2629440, 56)

# pd.set_option("display.max_columns", None)
# df
# 
INPUTS = schema['inputs']          # 입력 39개 (직접 고르지 말고 이 목록을 쓰기)
TARGET = 'rentals_target_1h'       # 반납 모델은 'returns_target_1h'

use = df.coverage_status.eq('observed_operating') & df[TARGET].notna()
train = df[use & df.split.eq('train')]
valid = df[use & df.split.eq('validation')]
test  = df[use & df.split.eq('test')]      # 마지막까지 열어보지 않기
print(len(train), len(valid), len(test))

(2629440, 56)
1569378 526178 524875


In [2]:
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib

print("🚀 Executing Self-Contained XGBoost Pipeline...")

# =====================================================================
# MODULE INITIALIZATION & MAPPING
# =====================================================================
INPUTS = schema['inputs']
TARGETS = ['rentals_target_1h', 'returns_target_1h']

# Step 2: Time-series masks extraction
use = df.coverage_status.eq('observed_operating') & df['rentals_target_1h'].notna() & df['returns_target_1h'].notna()
train_mask = use & df.split.eq('train')
valid_mask = use & df.split.eq('validation')
test_mask  = use & df.split.eq('test')

# Generate isolated coordinate points to align subsets accurately
X_valid_idx = df[valid_mask].index
X_test_idx  = df[test_mask].index

# FIXED: Explicit definition of the core evaluation score matrix
def score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None) # Clip negative numbers to 0
    err = y_pred - y_true
    total = np.abs(y_true).sum()
    
    mae = np.abs(err).mean()
    rmse = np.sqrt((err**2).mean())
    wape = np.abs(err).sum() / total if total else np.nan
    bias = err.mean()
    
    return {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'Bias': bias}

# FIXED: Explicit segment definitions mapped directly to your dataset labels
segments = {
    '전체': (X_valid_idx, X_test_idx),
    
    '출근': (X_valid_idx[df.loc[X_valid_idx, 'commute_period'] == 'morning'], 
            X_test_idx[df.loc[X_test_idx, 'commute_period'] == 'morning']),
    
    '퇴근': (X_valid_idx[df.loc[X_valid_idx, 'commute_period'] == 'evening'], 
            X_test_idx[df.loc[X_test_idx, 'commute_period'] == 'evening']),
    
    '역세권': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '역세권'], 
              X_test_idx[df.loc[X_test_idx, 'station_group'] == '역세권']),
    
    '업무지구': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '업무지구'], 
                X_test_idx[df.loc[X_test_idx, 'station_group'] == '업무지구']),
    
    '주거지': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '주거지'], 
              X_test_idx[df.loc[X_test_idx, 'station_group'] == '주거지'])
}

metrics_records = []
pred_df_list = []

# =====================================================================
# MODEL TRAINING & PROCESSING LOOPS
# =====================================================================
for target in TARGETS:
    print(f"\n🚀 Training XGBoost Model for target variable: {target}...")
    
    X_train, y_train = df.loc[train_mask, INPUTS].copy(), df.loc[train_mask, target]
    X_valid, y_valid = df.loc[valid_mask, INPUTS].copy(), df.loc[valid_mask, target]
    X_test, y_test   = df.loc[test_mask, INPUTS].copy(), df.loc[test_mask, target]
    
    # Categorical string conversion for native XGBoost handling
    categorical_cols = ['station_group', 'region', 'commute_period', 'holiday_type']
    for col in categorical_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col].astype('category')
            X_valid[col] = X_valid[col].astype('category')
            X_test[col] = X_test[col].astype('category')

    # Initialize XGBoost matching reproducibility constraints (Seed 42)
    model = xgb.XGBRegressor(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        enable_categorical=True,
        tree_method="hist",  # Boosts execution efficiency across huge tables
        early_stopping_rounds=30
    )
    
    start_time = time.time()
    model.fit(
        X_train, y_train, 
        eval_set=[(X_valid, y_valid)], 
        verbose=False
    )
    train_seconds = time.time() - start_time
    print(f"⏱️ Model training completed in {train_seconds:.2f} seconds.")
    joblib.dump(model, f'./models/bicycle_xgboost_{target}.pkl')
    
    # Generate index-preserved predictions
    valid_pred_arr = np.clip(model.predict(X_valid), 0, None)
    test_pred_arr  = np.clip(model.predict(X_test), 0, None)
    
    valid_pred_series = pd.Series(valid_pred_arr, index=X_valid.index)
    test_pred_series  = pd.Series(test_pred_arr, index=X_test.index)
    
    # Run evaluation across segmented subsets
    for seg_name, (v_idx, t_idx) in segments.items():
        v_res = score(df.loc[v_idx, target], valid_pred_series.loc[v_idx])
        t_res = score(df.loc[t_idx, target], test_pred_series.loc[t_idx])
        
        metrics_records.append({
            'model': 'XGBoost', 'target': target, 'split': 'validation', 'segment': seg_name,
            'n': len(v_idx), 'MAE': v_res['MAE'], 'RMSE': v_res['RMSE'],
            'WAPE': v_res['WAPE'], 'Bias': v_res['Bias'], 'train_seconds': train_seconds
        })
        metrics_records.append({
            'model': 'XGBoost', 'target': target, 'split': 'test', 'segment': seg_name,
            'n': len(t_idx), 'MAE': t_res['MAE'], 'RMSE': t_res['RMSE'],
            'WAPE': t_res['WAPE'], 'Bias': t_res['Bias'], 'train_seconds': train_seconds
        })

    # Restructure prediction row logs for output file compatibility
    for split_name, mask, pred_vals in [('validation', valid_mask, valid_pred_arr), ('test', test_mask, test_pred_arr)]:
        sub_pred = df.loc[mask, ['station_id', 'timestamp']].copy()
        sub_pred['split'] = split_name
        sub_pred['target'] = target
        sub_pred['y_true'] = df.loc[mask, target].astype(float)
        sub_pred['y_pred'] = pred_vals.astype(float)
        pred_df_list.append(sub_pred)

# =====================================================================
# STEP 5: OUTPUT DELIVERY FILE WRITING
# =====================================================================
print("\n💾 Writing production outputs to Shared Drive directory path...")

DATA = Path('./results/')

metrics_df = pd.DataFrame(metrics_records)
metrics_df.to_csv(DATA / 'metrics_XGBoost.csv', index=False)

pred_df = pd.concat(pred_df_list, ignore_index=True)
pred_df.to_parquet(DATA / 'pred_XGBoost.parquet', index=False, engine='pyarrow')

params_dict = {
    "inputs": INPUTS,
    "hyperparameters": {
        "objective": "reg:squarederror",
        "learning_rate": 0.05,
        "max_depth": 6,
        "random_state": 42
    }
}
with open(DATA / 'params_XGBoost.json', 'w', encoding='utf-8') as f:
    json.dump(params_dict, f, indent=2, ensure_ascii=False)

print("🎯 Finished successfully! XGBoost deliverable pipeline complete with no dependency errors.")


🚀 Executing Self-Contained XGBoost Pipeline...

🚀 Training XGBoost Model for target variable: rentals_target_1h...
⏱️ Model training completed in 498.98 seconds.

🚀 Training XGBoost Model for target variable: returns_target_1h...
⏱️ Model training completed in 785.47 seconds.

💾 Writing production outputs to Shared Drive directory path...
🎯 Finished successfully! XGBoost deliverable pipeline complete with no dependency errors.


In [3]:
import pandas as pd

# 1. Load the generated metrics file from your DATA path
metrics_path = DATA / 'metrics_XGBoost.csv'
metrics_df = pd.read_csv(metrics_path)

# 2. Separate Validation and Test for clean viewing
val_summary = metrics_df[metrics_df['split'] == 'validation'].copy()
test_summary = metrics_df[metrics_df['split'] == 'test'].copy()

# 3. Display Validation Dashboard (Use this to compare against Team A's Baseline)
print("=================== 📊 XGBoost 검증 ===================")
val_pivot = val_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(val_pivot.round(4))

# 4. Display Final Test Dashboard (The 2025 Evaluation - Checked Once)
print("\n=================== 🏁 XGBoost 테스트 ===================")
test_pivot = test_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(test_pivot.round(4))


=================== 📊 XGBoost 검증 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.3173            1.3413            0.4563   
역세권                1.6970            1.6974            0.3900   
전체                 1.3966            1.4012            0.4410   
주거지                1.1755            1.1647            0.5197   
출근                 1.6670            1.7158            0.3930   
퇴근                 2.1363            2.0869            0.3529   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.4422            0.0313           -0.0025  
역세권                0.3807            0.0971            0.0309  
전체                 0.4297            0.0460            0.0162  
주거지                0.5084            0.0094            0.0202  
출근                 0.3447           -0.0731           -0.0893  
퇴근                 0.3475            0.1504            0.0564


=================== 🏁 XGBoost 테스트 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.1989            1.2120            0.5007   
역세권                1.5236            1.5301            0.4347   
전체                 1.2663            1.2704            0.4866   
주거지                1.0769            1.0695            0.5641   
출근                 1.5196            1.5687            0.4179   
퇴근                 1.9408            1.8851            0.3972   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.4794            0.1683            0.1215  
역세권                0.4226            0.2276            0.1491  
전체                 0.4726            0.1642            0.1296  
주거지                0.5579            0.0970            0.1181  
출근                 0.3599            0.0689            0.0218  
퇴근                 0.3983            0.4077            0.3432